# 第2回 演習: 用語集を LoRA で学習し、学習前後を比較する

用語集と、用語集にない語への回答例を含む SFT データでモデルを LoRA 学習し、学習前後を比べます。評価では見出し語・言い換え・未登録語を尋ね、JSON 形式、`needs_review`、回答内容を確認します。

**この演習で確かめること**

1. 用語集にある項目では `needs_review: false` と定義を回答できるか。
2. 質問の言い回しを変えても、用語集の定義に沿って JSON で回答できるか。
3. 用語集にない項目では、推測せず `needs_review: true` と回答できるか。
4. 発展: 同じモデルを 4bit 量子化で読み込むと、学習しなくても出力が変わる（セクション 8）。

`needs_review` は「モデル自身がその概念を知らない」という意味ではなく、「この用語集に記載がない」という意味で使います。評価用の未知語とは別の語を一部 SFT データに含め、未見の未知語への一般化を測ります。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、用語集と評価データ、学習前評価 | 約 5 分 |
| 4〜7 | SFT データの読み込みと学習、学習後評価、結果の提出 | 学習データ量に依存 |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。

**使うもの**: Google Colab（T4 GPU、無料枠）、`Qwen/Qwen3.5-0.8B`（Apache-2.0）、`data/glossary.json`、`glossary_sft.jsonl`。セクション 2 で後ろの 2 ファイルをアップロードします。

**読み方**: 各セクションの説明に「やること」「見るところ」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

版: 1.3（2026-11-02）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 3,   # 用語集 SFT データを 3 周学習（ステップ数はデータ件数に依存）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = True  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 用語集と評価データの準備

**やること**: `data/glossary.json` と `glossary_sft.jsonl` をアップロードし、用語集に沿った評価用の質問と正解を用意します。

評価用 `test` は 12 問です。用語集の見出し語、定義の言い換え、用語集にない語を各4問ずつ用意します。`gold` には JSON 形式の正解、`needs_review`、回答に必要な語句を保存し、生成時のモデル入力には正解を含めません。

学習データには用語集の項目（`needs_review: false`）に加え、評価とは異なる未登録語への例（`needs_review: true`）を一部含めています。評価用の未知語は学習例に含めず、未見の語への判定を測ります。`needs_review` は用語集への登録有無を表します。

**見るところ**: 用語集と評価ケースの件数・分類を確認してください。`test.jsonl` と `gold.json` はこのセルで作成します。

In [ ]:
from google.colab import files

uploaded = files.upload()
required_files = {"glossary.json", "glossary_sft.jsonl"}
missing_files = required_files - set(uploaded)
if missing_files:
    raise FileNotFoundError(f"アップロードされていないファイル: {', '.join(sorted(missing_files))}")

GLOSSARY_PATH = Path("glossary.json")
SFT_PATH = Path("glossary_sft.jsonl")
glossary_data = json.loads(GLOSSARY_PATH.read_text(encoding="utf-8"))
terms = glossary_data["terms"]
terms_by_id = {term["id"]: term for term in terms}
registered_terms = ", ".join(term["term"] for term in terms)
SYSTEM = (
    "あなたは講座用語集の回答係です。必ずJSONオブジェクトだけを1行で返し、キーはanswer（文字列）とneeds_review（真偽値）の2つだけにしてください。"
    "用語集にある項目は定義をanswerに入れ、needs_reviewはfalseにします。用語集にない項目は推測せず、answerを『用語集には記載がありません。』、needs_reviewをtrueにします。"
    "needs_reviewはモデルの一般知識ではなく、この用語集に項目が登録されているかを示します。"
    f"登録されている見出し語: {registered_terms}"
)

# 用語集の見出し語をそのまま尋ねる質問、言い換え質問、用語集にない語の質問。
# answer は用語集の定義（または用語集にない場合の正解）を参照する。
TEST_CASES = [
    {"category": "見出し語", "question": "LoRAの定義を教えてください。", "term_id": "lora",
     "must_include": ["事前学習済みの重み", "低ランク行列"]},
    {"category": "見出し語", "question": "コンテキストウィンドウの意味は何ですか？", "term_id": "context-window",
     "must_include": ["入力と出力", "合計トークン数"]},
    {"category": "見出し語", "question": "ハルシネーションを説明してください。", "term_id": "hallucination",
     "must_include": ["事実と異なる内容", "もっともらしく生成"]},
    {"category": "見出し語", "question": "レート制限とはどのような仕組みですか？", "term_id": "rate-limit",
     "must_include": ["一定時間あたり", "HTTP 429"]},
    {"category": "言い換え", "question": "事前学習済みの重みを固定し、2つの小さな行列で更新分を表す方法は何ですか？", "term_id": "lora",
     "must_include": ["事前学習済みの重み", "低ランク行列"]},
    {"category": "言い換え", "question": "1回の呼び出しで入力や会話履歴、返答を合わせて扱えるトークン数の上限は何ですか？", "term_id": "context-window",
     "must_include": ["入力と出力", "合計トークン数"]},
    {"category": "言い換え", "question": "実際には誤りなのに自然で本当らしく見える内容をモデルが作る現象は？", "term_id": "hallucination",
     "must_include": ["事実と異なる内容", "もっともらしく生成"]},
    {"category": "言い換え", "question": "一定時間内の呼び出し上限を超えて HTTP 429 が返ることに関係する用語は？", "term_id": "rate-limit",
     "must_include": ["一定時間あたり", "HTTP 429"]},
    {"category": "用語集にない語", "question": "RAG（検索拡張生成）とは何ですか？", "term_id": None,
     "must_include": ["用語集には記載がありません"]},
    {"category": "用語集にない語", "question": "ベクトルデータベースの役割を説明してください。", "term_id": None,
     "must_include": ["用語集には記載がありません"]},
    {"category": "用語集にない語", "question": "埋め込みベクトル（embedding）とは何ですか？", "term_id": None,
     "must_include": ["用語集には記載がありません"]},
    {"category": "用語集にない語", "question": "プロンプトインジェクションへの対策を教えてください。", "term_id": None,
     "must_include": ["用語集には記載がありません"]},
]

unknown_terms = ("RAG", "ベクトルデータベース", "埋め込みベクトル", "プロンプトインジェクション")
known_names = {name.casefold() for term in terms for name in [term["term"], *term.get("aliases", [])]}
assert all(name.casefold() not in known_names for name in unknown_terms), "用語集にある語を未知語として指定しています"

def make_example(case):
    return {"question": case["question"], "category": case["category"]}

test = [make_example(case) for case in TEST_CASES]
gold = {
    case["question"]: {
        "category": case["category"],
        "is_in_glossary": case["term_id"] is not None,
        "needs_review": case["term_id"] is None,
        "reference_answer": terms_by_id[case["term_id"]]["definition"] if case["term_id"] else "用語集には記載がありません。",
        "must_include": case["must_include"],
    }
    for case in TEST_CASES
}
assert len({example["question"] for example in test}) == len(test), "評価質問が重複しています"
assert len(test) == 12 and len(gold) == len(test)
print(f"用語集={len(terms)} 件 | test={len(test)} 件 | gold={len(gold)} 件")
print({category: sum(example["category"] == category for example in test) for category in sorted({example["category"] for example in test})})
(WORK / "test.jsonl").write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in test) + "\n", encoding="utf-8")
(WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、セクション 2 の12問に回答させます。回答をJSONとして解析し、正確なキーと型があるか（形式）、`needs_review` と回答中の正解語句が期待どおりか（内容）を別々に採点します。

**見るところ**: 1つ目のセルではモデルの読み込みと重みのメモリ量を確認します。次のセルでは形式正解率、内容正解率、両方を満たす率を確認します。質問分類別の結果と回答原文も表示され、`/content/qlora_lab/eval_before.json` に保存されます。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: 公開モデルはログインしなくてもダウンロードできます。ログインが必要なのは利用規約への同意が必要なモデルや非公開リポジトリを使う場合などです。
- `... is falling back to its reference PyTorch implementation ...`: 一部の層に高速な実装が入っていないため、遅いが正しい実装で動くという案内です。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)


def build_messages(example):
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": example["question"]},
    ]


def generate(model, example):
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def check_format(text: str):
    """JSONとして読め、answer/needs_reviewのキーと型が正確かを返す。"""
    try:
        obj = json.loads(text)
    except json.JSONDecodeError as exc:
        return False, f"json_error: {exc.msg}", None
    if not isinstance(obj, dict):
        return False, "not_object", None
    if set(obj) != {"answer", "needs_review"}:
        return False, "wrong_keys", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool):
        return False, "wrong_type", obj
    return True, "", obj


def check_content(obj, gold_entry: dict) -> bool:
    """needs_reviewとanswer内の必須語句が正解に一致するかを返す。"""
    if not isinstance(obj, dict) or not {"answer", "needs_review"}.issubset(obj):
        return False
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool):
        return False
    return obj["needs_review"] == gold_entry["needs_review"] and all(
        required in obj["answer"] for required in gold_entry["must_include"]
    )


def mark(ok: bool) -> str:
    return "○" if ok else "×"


def evaluate(model, label: str):
    """testの回答を形式・内容で別々に採点し、原文を保存する。"""
    rows = []
    t0 = time.time()
    for example in test:
        raw = generate(model, example)
        expected = gold[example["question"]]
        format_ok, format_reason, obj = check_format(raw)
        content_ok = check_content(obj, expected)
        rows.append({
            "question": example["question"], "category": expected["category"], "raw": raw,
            "format_ok": format_ok, "format_reason": format_reason,
            "content_ok": content_ok, "both_ok": format_ok and content_ok,
            "gold_needs_review": expected["needs_review"],
            "reference_answer": expected["reference_answer"],
        })
    n = len(rows)
    category_rates = {
        category: {
            key: sum(row[key] for row in rows if row["category"] == category)
                 / sum(row["category"] == category for row in rows)
            for key in ("format_ok", "content_ok", "both_ok")
        }
        for category in sorted({row["category"] for row in rows})
    }
    summary = {
        "label": label, "n": n,
        "format_rate": sum(row["format_ok"] for row in rows) / n,
        "content_rate": sum(row["content_ok"] for row in rows) / n,
        "both_rate": sum(row["both_ok"] for row in rows) / n,
        "category_rates": category_rates,
        "seconds": round(time.time() - t0, 1),
    }
    (WORK / f"eval_{label}.json").write_text(
        json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows


before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{
    "分類": row["category"], "質問": row["question"], "正解のneeds_review": row["gold_needs_review"],
    "正解例": row["reference_answer"], "学習前の出力": row["raw"],
    "形式": mark(row["format_ok"]), "形式エラー": row["format_reason"], "内容": mark(row["content_ok"]),
} for row in before_rows]))

## 4. LoRA 学習

**やること**: `glossary_sft.jsonl` の各行を JSON として読み、各行の `messages` を加工せず `Dataset.from_list` に渡します。データセットを固定 seed で学習用と検証用に分け、セクション 4 の `SFTTrainer` で学習します。

**見るところ**: 学習可能なパラメータ数、学習ステップ数、学習時間を確認します。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

records = [json.loads(line) for line in SFT_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
if not records or any(set(record) != {"messages"} for record in records):
    raise ValueError("glossary_sft.jsonl は messages キーを持つ JSONL にしてください")
known_definitions = {term["definition"] for term in terms}
training_questions = set()
needs_review_counts = {False: 0, True: 0}
for record in records:
    messages = record["messages"]
    if len(messages) != 2 or [message.get("role") for message in messages] != ["user", "assistant"]:
        raise ValueError("各SFT例はuserとassistantの2つのmessagesにしてください")
    question = messages[0]["content"]
    if question in training_questions or question in {example["question"] for example in test}:
        raise ValueError(f"学習質問が重複または評価質問と一致しています: {question}")
    training_questions.add(question)
    try:
        target = json.loads(messages[1]["content"])
    except json.JSONDecodeError as exc:
        raise ValueError(f"assistant回答がJSONではありません: {question}") from exc
    if not isinstance(target, dict) or set(target) != {"answer", "needs_review"}:
        raise ValueError(f"assistant回答のJSONキーが不正です: {question}")
    if not isinstance(target["answer"], str) or not isinstance(target["needs_review"], bool):
        raise ValueError(f"assistant回答のJSON型が不正です: {question}")
    if target["needs_review"]:
        if target["answer"] != "用語集には記載がありません。":
            raise ValueError(f"未登録語の回答が不正です: {question}")
    elif target["answer"] not in known_definitions:
        raise ValueError(f"登録語の回答が用語集の定義と一致しません: {question}")
    needs_review_counts[target["needs_review"]] += 1
if not all(needs_review_counts.values()):
    raise ValueError("学習データには登録語と未登録語の両方の例が必要です")
print(f"学習例: needs_review=false {needs_review_counts[False]}件 | true {needs_review_counts[True]}件")
dataset = Dataset.from_list(records)
split = dataset.train_test_split(test_size=0.1, seed=CONFIG["seed"])
train_ds = split["train"]
valid_ds = split["test"]
print(f"SFT データ: 全{len(dataset)}件 | train={len(train_ds)} | valid={len(valid_ds)}")

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから設定済みのエポック数まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師の実行結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じtest・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前と同じ12問・同じ生成設定で評価し、前後を並べます。変えたのはLoRA adapterの有無だけです。

**見るところ**

- 比較表: JSON形式の正解率、内容（`needs_review`と回答語句）の正解率、両方を満たした率を比較します。
- `退行`: 学習前は形式と内容の両方に正解し、学習後に不正解となった質問です。
- 質問ごとの表: 正解例と回答原文を見比べ、JSONの形式、`needs_review`、回答語句のどこで失敗したかを確認してください。

未登録語の学習例と評価例は異なるため、未見の語で `needs_review: true` と判定できるかも確認します。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式・内容の両方に正解したのに、学習後に不正解となった質問
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

detail = pd.DataFrame([{
    "分類": b["category"], "質問": b["question"], "正解のneeds_review": b["gold_needs_review"],
    "正解例": b["reference_answer"], "学習前の出力": b["raw"],
    "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
    "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}",
} for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. 評価結果の読み方

- **形式**: JSONとして読め、キーが `answer` と `needs_review` のちょうど2つで、値の型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解語句が `answer` に含まれること。

用語集にある質問の正解は `needs_review: false`、用語集にない質問の正解は `true` です。指定語句にない同義表現は不正解になることがあるため、形式・内容の率だけでなく回答原文も確認してください。

## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、学習後に形式または内容で不正解となった質問を `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`、`test.jsonl`、`gold.json`（`/content/qlora_lab` にあります。`USE_DRIVE = True` なら主要な結果ファイルはDriveにもコピーされます）
- セクション5の比較表と、形式・内容のどちらで失敗したかの説明
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "executed_training_myself": True,  # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。

- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。導入に 1 分ほどかかります。
- 4bit の読み込みは `device_map="auto"` で GPU に配置します（bitsandbytes の読み込み方式に合わせています）。
- 4bit 側で学習まで試す場合は、`prepare_model_for_kbit_training(model_4bit)` を通してからセクション 4 と同じ手順で学習し、学習前後とも 4bit の基盤モデルで評価します。

**見るところ**

- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。
- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。分類別の正解率と回答原文を確認してください。

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"分類": r["category"], "質問": r["question"], "正解のneeds_review": r["gold_needs_review"],
                       "正解例": r["reference_answer"], "4bit・学習前の出力": r["raw"],
                       "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式エラー": r["format_reason"]} for r in q_rows]))